In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
import os, time

BASE_PATH = '/content/drive/My Drive/phishing project datasets/processed'
EMBED_PATH = f'{BASE_PATH}/embeddings'
os.makedirs(EMBED_PATH, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

train_df = pd.read_csv(f'{BASE_PATH}/train_features.csv')
test_df = pd.read_csv(f'{BASE_PATH}/test_features.csv')
print(train_df.shape, test_df.shape)

Mounted at /content/drive
Using device: cuda
(202583, 36) (50646, 36)


In [ ]:
enron_raw = pd.read_csv(f'{BASE_PATH}/enron_emails_cleaned_temporal.csv')
nazario_raw = pd.read_csv(f'{BASE_PATH}/nazario5_cleaned_temporal.csv')

# Enron: email_id was the original message_id
enron_subjects = enron_raw[['message_id', 'subject']].rename(columns={'message_id': 'email_id'})

# Nazario: email_id was a composite of sender_norm + utc_datetime -- rebuild identically
nazario_raw['utc_datetime'] = pd.to_datetime(nazario_raw['utc_datetime'], errors='coerce')
nazario_raw['sender_norm'] = nazario_raw['sender_address'].astype(str).str.strip().str.lower()
nazario_raw['email_id'] = nazario_raw['sender_norm'] + '_' + nazario_raw['utc_datetime'].astype(str)
nazario_subjects = nazario_raw[['email_id', 'subject']]

subjects_all = pd.concat([enron_subjects, nazario_subjects], ignore_index=True).drop_duplicates(subset='email_id')

train_df = train_df.merge(subjects_all, on='email_id', how='left')
test_df = test_df.merge(subjects_all, on='email_id', how='left')

print("Train missing subjects:", train_df['subject'].isna().sum(), "/", len(train_df))
print("Test missing subjects:", test_df['subject'].isna().sum(), "/", len(test_df))

Train missing subjects: 6852 / 202583
Test missing subjects: 1731 / 50646


In [ ]:
train_df['model_input_text'] = "[SUBJECT] " + train_df['subject'].fillna('') + " [BODY] " + train_df['body_text'].fillna('')
test_df['model_input_text'] = "[SUBJECT] " + test_df['subject'].fillna('') + " [BODY] " + test_df['body_text'].fillna('')

print(train_df['model_input_text'].iloc[0][:300])

[SUBJECT] RE: [BODY] That fuckin' SUCKS!  Call 'em and tell 'em you are suing!!!!!!!!!!!!!!!!!!!!!!!

 -----Original Message-----
From: 	Marc Stewart <Stewart@Mallia.com>@ENRON  
Sent:	Tuesday, February 05, 2002 11:18 AM
To:	Baughman Jr., Don
Subject:	 

aahhhh!

my computer shipping date has been "


In [ ]:
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

sample = train_df['model_input_text'].sample(5000, random_state=42)
lengths = sample.apply(lambda x: len(tokenizer.encode(x, truncation=False)))

print(lengths.describe())
print("95th percentile:", lengths.quantile(0.95))

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1136 > 512). Running this sequence through the model will result in indexing errors


count     5000.000000
mean       492.906600
std       1643.260916
min          9.000000
25%         88.000000
50%        218.000000
75%        492.000000
max      69496.000000
Name: model_input_text, dtype: float64
95th percentile: 1601.1000000000004


In [ ]:
MAX_LENGTH = 512
CHUNK_SIZE = 5000   # rows per checkpointed chunk
BATCH_SIZE = 32     # forward-pass batch size within a chunk

model = AutoModel.from_pretrained('distilbert-base-uncased').to(device)
model.eval()
if device.type == 'cuda':
    model = model.half()  # FP16 for speed/memory on T4

print("Model loaded on", device, "| dtype:", next(model.parameters()).dtype)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded on cuda | dtype: torch.float16


In [ ]:
@torch.no_grad()
def embed_batch(texts):
    encoded = tokenizer(
        texts, padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors='pt'
    ).to(device)

    if device.type == 'cuda':
        encoded = {k: v for k, v in encoded.items()}  # input_ids/attention_mask stay int, fine as-is

    outputs = model(**encoded)
    last_hidden = outputs.last_hidden_state  # (batch, seq_len, hidden_dim)

    # mean pooling, masked so padding tokens don't count
    mask = encoded['attention_mask'].unsqueeze(-1).expand(last_hidden.size()).float()
    summed = torch.sum(last_hidden * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    mean_pooled = summed / counts

    return mean_pooled.float().cpu().numpy()

In [ ]:
def process_split(df, split_name):
    manifest_path = f'{EMBED_PATH}/{split_name}_manifest.txt'
    completed_chunks = set()
    if os.path.exists(manifest_path):
        with open(manifest_path) as f:
            completed_chunks = set(line.strip() for line in f)

    n_chunks = int(np.ceil(len(df) / CHUNK_SIZE))
    print(f"{split_name}: {len(df)} rows, {n_chunks} chunks, {len(completed_chunks)} already done")

    for chunk_idx in range(n_chunks):
        chunk_id = f"chunk_{chunk_idx:04d}"
        out_file = f'{EMBED_PATH}/{split_name}_{chunk_id}.npz'

        if chunk_id in completed_chunks and os.path.exists(out_file):
            continue  # already processed, skip -- this is what makes reruns resumable

        start = chunk_idx * CHUNK_SIZE
        end = min(start + CHUNK_SIZE, len(df))
        chunk_df = df.iloc[start:end]

        chunk_embeddings = []
        texts = chunk_df['model_input_text'].tolist()

        for b_start in range(0, len(texts), BATCH_SIZE):
            batch_texts = texts[b_start:b_start + BATCH_SIZE]
            emb = embed_batch(batch_texts)
            chunk_embeddings.append(emb)

        chunk_embeddings = np.vstack(chunk_embeddings)
        chunk_ids = chunk_df['email_id'].values

        np.savez(out_file, email_id=chunk_ids, embeddings=chunk_embeddings)

        with open(manifest_path, 'a') as f:
            f.write(chunk_id + '\n')

        if chunk_idx % 5 == 0:
            print(f"  {split_name} {chunk_id} done ({end}/{len(df)})")

    print(f"{split_name} complete.")

In [ ]:
t0 = time.time()
process_split(train_df, 'train')
print(f"Train embedding time: {(time.time()-t0)/60:.1f} min")

train: 202583 rows, 41 chunks, 0 already done
  train chunk_0000 done (5000/202583)
  train chunk_0005 done (30000/202583)
  train chunk_0010 done (55000/202583)
  train chunk_0015 done (80000/202583)
  train chunk_0020 done (105000/202583)
  train chunk_0025 done (130000/202583)
  train chunk_0030 done (155000/202583)
  train chunk_0035 done (180000/202583)
  train chunk_0040 done (202583/202583)
train complete.
Train embedding time: 15.2 min


In [ ]:
t0 = time.time()
process_split(test_df, 'test')
print(f"Test embedding time: {(time.time()-t0)/60:.1f} min")

test: 50646 rows, 11 chunks, 0 already done
  test chunk_0000 done (5000/50646)
  test chunk_0005 done (30000/50646)
  test chunk_0010 done (50646/50646)
test complete.
Test embedding time: 4.1 min


In [ ]:
def consolidate_embeddings(df, split_name):
    chunk_files = sorted([
        f for f in os.listdir(EMBED_PATH)
        if f.startswith(f'{split_name}_chunk_') and f.endswith('.npz')
    ])
    print(f"{split_name}: found {len(chunk_files)} chunk files")

    all_ids, all_embeddings = [], []
    for f in chunk_files:
        data = np.load(f'{EMBED_PATH}/{f}', allow_pickle=True)
        all_ids.append(data['email_id'])
        all_embeddings.append(data['embeddings'])

    ids_concat = np.concatenate(all_ids)
    emb_concat = np.vstack(all_embeddings)

    # reorder to match the original df row order exactly (chunk write order should already match, this just guarantees it)
    id_to_pos = {eid: i for i, eid in enumerate(ids_concat)}
    order = [id_to_pos[eid] for eid in df['email_id'].values]

    return ids_concat[order], emb_concat[order]

train_ids, train_embeddings = consolidate_embeddings(train_df, 'train')
test_ids, test_embeddings = consolidate_embeddings(test_df, 'test')

print("Train embeddings shape:", train_embeddings.shape)
print("Test embeddings shape:", test_embeddings.shape)

train: found 41 chunk files
test: found 11 chunk files
Train embeddings shape: (202583, 768)
Test embeddings shape: (50646, 768)


In [ ]:
print("Train NaNs:", np.isnan(train_embeddings).sum())
print("Test NaNs:", np.isnan(test_embeddings).sum())
print("IDs match df order (train):", (train_ids == train_df['email_id'].values).all())
print("IDs match df order (test):", (test_ids == test_df['email_id'].values).all())
print("Embedding norm stats (train):", np.linalg.norm(train_embeddings, axis=1).mean(), "+/-", np.linalg.norm(train_embeddings, axis=1).std())

Train NaNs: 0
Test NaNs: 0
IDs match df order (train): True
IDs match df order (test): True
Embedding norm stats (train): 7.76347 +/- 0.5036558


In [ ]:
np.save(f'{EMBED_PATH}/train_embeddings_final.npy', train_embeddings)
np.save(f'{EMBED_PATH}/test_embeddings_final.npy', test_embeddings)
pd.DataFrame({'email_id': train_ids}).to_csv(f'{EMBED_PATH}/train_embeddings_index.csv', index=False)
pd.DataFrame({'email_id': test_ids}).to_csv(f'{EMBED_PATH}/test_embeddings_index.csv', index=False)

print("Saved consolidated embeddings + index files.")

Saved consolidated embeddings + index files.


transformer tuning


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q datasets

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import gc, os, glob

from transformers import (
    AutoTokenizer, DistilBertForSequenceClassification,
    DataCollatorWithPadding, TrainingArguments, Trainer
)
from datasets import Dataset
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score, average_precision_score

BASE_PATH = '/content/drive/My Drive/phishing project datasets/processed'
MODEL_CHECKPOINT_DIR = f'{BASE_PATH}/finetune_checkpoints'
os.makedirs(MODEL_CHECKPOINT_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

MAX_LENGTH = 256
TRAIN_BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 8
NUM_EPOCHS = 1

Mounted at /content/drive
Using device: cuda


In [ ]:
train_meta = pd.read_csv(f'{BASE_PATH}/train_features.csv', usecols=['email_id', 'label', 'body_text'])
test_meta = pd.read_csv(f'{BASE_PATH}/test_features.csv', usecols=['email_id', 'label', 'body_text'])

enron_subj = pd.read_csv(f'{BASE_PATH}/enron_emails_cleaned_temporal.csv', usecols=['message_id', 'subject'])
enron_subj = enron_subj.rename(columns={'message_id': 'email_id'})

nazario_raw = pd.read_csv(
    f'{BASE_PATH}/nazario5_cleaned_temporal.csv',
    usecols=['sender_address', 'utc_datetime', 'subject']
)
nazario_raw['utc_datetime'] = pd.to_datetime(nazario_raw['utc_datetime'], errors='coerce')
nazario_raw['sender_norm'] = nazario_raw['sender_address'].astype(str).str.strip().str.lower()
nazario_raw['email_id'] = nazario_raw['sender_norm'] + '_' + nazario_raw['utc_datetime'].astype(str)
nazario_subj = nazario_raw[['email_id', 'subject']]

subjects_all = pd.concat([enron_subj, nazario_subj], ignore_index=True).drop_duplicates(subset='email_id')

train_meta = train_meta.merge(subjects_all, on='email_id', how='left')
test_meta = test_meta.merge(subjects_all, on='email_id', how='left')

del enron_subj, nazario_raw, nazario_subj, subjects_all
gc.collect()

print(train_meta.shape, test_meta.shape)

(202583, 4) (50646, 4)


In [ ]:
train_meta['text'] = "[SUBJECT] " + train_meta['subject'].fillna('') + " [BODY] " + train_meta['body_text'].fillna('')
test_meta['text'] = "[SUBJECT] " + test_meta['subject'].fillna('') + " [BODY] " + test_meta['body_text'].fillna('')

train_meta = train_meta[['label', 'text']]
test_meta = test_meta[['label', 'text']]

gc.collect()
print(train_meta.iloc[0]['text'][:200])

[SUBJECT] RE: [BODY] That fuckin' SUCKS!  Call 'em and tell 'em you are suing!!!!!!!!!!!!!!!!!!!!!!!

 -----Original Message-----
From: 	Marc Stewart <Stewart@Mallia.com>@ENRON  
Sent:	Tuesday, Februa


In [ ]:
label_counts = train_meta['label'].value_counts().sort_index()
total = label_counts.sum()
class_weights = torch.tensor([total / (2 * c) for c in label_counts], dtype=torch.float).to(device)
print("Class weights [benign, phishing]:", class_weights)

Class weights [benign, phishing]: tensor([ 0.5060, 42.4880], device='cuda:0')


In [ ]:
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

train_ds = Dataset.from_pandas(train_meta[['text', 'label']], preserve_index=False)
test_ds = Dataset.from_pandas(test_meta[['text', 'label']], preserve_index=False)

del train_meta, test_meta
gc.collect()

def tokenize_fn(batch):
    return tokenizer(batch['text'], truncation=True, max_length=MAX_LENGTH)

train_ds = train_ds.map(tokenize_fn, batched=True, batch_size=1000, remove_columns=['text'])
test_ds = test_ds.map(tokenize_fn, batched=True, batch_size=1000, remove_columns=['text'])

train_ds = train_ds.rename_column('label', 'labels')
test_ds = test_ds.rename_column('label', 'labels')

# Intentionally no .set_format('torch') -- that formatter triggers a broken
# torchvision.io.VideoReader import in this environment. DataCollatorWithPadding
# handles list -> tensor conversion itself via tokenizer.pad(), a separate path.

print(train_ds)
print(test_ds)
print(train_ds[0])

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/202583 [00:00<?, ? examples/s]

Map:   0%|          | 0/50646 [00:00<?, ? examples/s]

Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 202583
})
Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 50646
})
{'labels': 0, 'input_ids': [101, 1031, 3395, 1033, 2128, 1024, 1031, 2303, 1033, 2008, 6616, 2378, 1005, 19237, 999, 2655, 1005, 7861, 1998, 2425, 1005, 7861, 2017, 2024, 24086, 3070, 999, 999, 999, 999, 999, 999, 999, 999, 999, 999, 999, 999, 999, 999, 999, 999, 999, 999, 999, 999, 999, 999, 999, 1011, 1011, 1011, 1011, 1011, 2434, 4471, 1011, 1011, 1011, 1011, 1011, 2013, 1024, 7871, 5954, 1026, 5954, 1030, 6670, 2401, 1012, 4012, 1028, 1030, 4372, 4948, 2741, 1024, 9857, 1010, 2337, 5709, 1010, 2526, 2340, 1024, 2324, 2572, 2000, 1024, 8670, 8953, 2386, 3781, 1012, 1010, 2123, 3395, 1024, 9779, 23644, 23644, 999, 2026, 3274, 7829, 3058, 2038, 2042, 1000, 8001, 1000, 2011, 12418, 1012, 2612, 1997, 7829, 2651, 1010, 2009, 1005, 1055, 2000, 2911, 2006, 6185, 1013, 2260, 10

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(weight=class_weights)
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=1)[:, 1].numpy()
    preds = logits.argmax(axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary', zero_division=0)
    return {
        'precision': precision, 'recall': recall, 'f1': f1,
        'roc_auc': roc_auc_score(labels, probs),
        'pr_auc': average_precision_score(labels, probs)
    }

In [ ]:
model_finetune = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2).to(device)

training_args = TrainingArguments(
    output_dir=MODEL_CHECKPOINT_DIR,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    num_train_epochs=NUM_EPOCHS,
    fp16=torch.cuda.is_available(),
    eval_strategy="steps",
    eval_steps=2000,
    save_strategy="steps",
    save_steps=2000,
    save_total_limit=2,
    logging_steps=200,
    load_best_model_at_end=True,
    metric_for_best_model="pr_auc",
    report_to="none",
    dataloader_num_workers=0,
)

trainer = WeightedTrainer(
    model=model_finetune,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
existing_checkpoints = glob.glob(f'{MODEL_CHECKPOINT_DIR}/checkpoint-*')
resume = bool(existing_checkpoints)
print("Resuming from checkpoint:", resume)

trainer.train(resume_from_checkpoint=resume if resume else None)

Resuming from checkpoint: False


Step,Training Loss,Validation Loss,Precision,Recall,F1,Roc Auc,Pr Auc
2000,0.395481,0.219516,0.891791,0.802013,0.844523,0.966104,0.870805
4000,0.370792,0.210003,0.966535,0.823826,0.889493,0.970865,0.894999
6000,0.467253,0.166244,0.965049,0.833893,0.894689,0.979008,0.910587
6331,0.375936,0.159919,0.963532,0.842282,0.898836,0.979533,0.912114


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=6331, training_loss=0.6401650985069438, metrics={'train_runtime': 2084.8696, 'train_samples_per_second': 97.168, 'train_steps_per_second': 3.037, 'total_flos': 1.309399098377868e+16, 'train_loss': 0.6401650985069438, 'epoch': 1.0})

In [ ]:
FINAL_MODEL_DIR = f'{BASE_PATH}/baseline_distilbert_finetuned'
trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)
print(f"Saved model + tokenizer to: {FINAL_MODEL_DIR}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved model + tokenizer to: /content/drive/My Drive/phishing project datasets/processed/baseline_distilbert_finetuned


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

predictions = trainer.predict(test_ds)
logits = predictions.predictions
labels = predictions.label_ids
preds = logits.argmax(axis=1)

print(classification_report(labels, preds, target_names=['benign', 'phishing'], digits=4))
print("\nConfusion matrix:")
print(confusion_matrix(labels, preds))

              precision    recall  f1-score   support

      benign     0.9981    0.9996    0.9989     50050
    phishing     0.9635    0.8423    0.8988       596

    accuracy                         0.9978     50646
   macro avg     0.9808    0.9210    0.9489     50646
weighted avg     0.9977    0.9978    0.9977     50646


Confusion matrix:
[[50031    19]
 [   94   502]]


In [ ]:
from sklearn.metrics import precision_recall_curve

probs = torch.softmax(torch.tensor(logits), dim=1)[:, 1].numpy()
precisions, recalls, thresholds = precision_recall_curve(labels, probs)

# check a few candidate thresholds below the default 0.5
for t in [0.5, 0.3, 0.2, 0.1, 0.05]:
    preds_t = (probs >= t).astype(int)
    tp = ((preds_t == 1) & (labels == 1)).sum()
    fp = ((preds_t == 1) & (labels == 0)).sum()
    fn = ((preds_t == 0) & (labels == 1)).sum()
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0
    print(f"threshold={t:.2f}  precision={prec:.4f}  recall={rec:.4f}  caught={tp}/596  false_positives={fp}")

threshold=0.50  precision=0.9635  recall=0.8423  caught=502/596  false_positives=19
threshold=0.30  precision=0.9581  recall=0.8440  caught=503/596  false_positives=22
threshold=0.20  precision=0.9582  recall=0.8456  caught=504/596  false_positives=22
threshold=0.10  precision=0.9513  recall=0.8523  caught=508/596  false_positives=26
threshold=0.05  precision=0.9360  recall=0.8591  caught=512/596  false_positives=35


In [ ]:
FINAL_MODEL_DIR = f'{BASE_PATH}/baseline_distilbert_finetuned'
trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

import json
decision_config = {
    "chosen_threshold": 0.10,
    "rationale": "Trades 7 extra false positives (19->26) for 6 more caught phishing emails (502->508) vs default 0.5",
    "test_metrics_at_threshold": {"precision": 0.9513, "recall": 0.8523, "caught": "508/596", "false_positives": 26}
}
with open(f'{BASE_PATH}/model_decision_config.json', 'w') as f:
    json.dump(decision_config, f, indent=2)

print("Model, tokenizer, and decision config saved to Drive.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model, tokenizer, and decision config saved to Drive.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import torch
import json, gc

from transformers import AutoTokenizer, DistilBertForSequenceClassification
from datasets import Dataset
from sklearn.metrics import classification_report, confusion_matrix

BASE_PATH = '/content/drive/My Drive/phishing project datasets/processed'
FINAL_MODEL_DIR = f'{BASE_PATH}/baseline_distilbert_finetuned'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = DistilBertForSequenceClassification.from_pretrained(FINAL_MODEL_DIR).to(device)
tokenizer = AutoTokenizer.from_pretrained(FINAL_MODEL_DIR)
model.eval()

with open(f'{BASE_PATH}/model_decision_config.json') as f:
    decision_config = json.load(f)
print(decision_config)

MAX_LENGTH = 256

Mounted at /content/drive


Loading weights:   0%|          | 0/104 [00:03<?, ?it/s]

{'chosen_threshold': 0.1, 'rationale': 'Trades 7 extra false positives (19->26) for 6 more caught phishing emails (502->508) vs default 0.5', 'test_metrics_at_threshold': {'precision': 0.9513, 'recall': 0.8523, 'caught': '508/596', 'false_positives': 26}}


In [2]:
test_full = pd.read_csv(f'{BASE_PATH}/test_features.csv')

nazario_raw = pd.read_csv(
    f'{BASE_PATH}/nazario5_cleaned_temporal.csv',
    usecols=['sender_address', 'utc_datetime', 'subject']
)
nazario_raw['utc_datetime'] = pd.to_datetime(nazario_raw['utc_datetime'], errors='coerce')
nazario_raw['sender_norm'] = nazario_raw['sender_address'].astype(str).str.strip().str.lower()
nazario_raw['email_id'] = nazario_raw['sender_norm'] + '_' + nazario_raw['utc_datetime'].astype(str)
nazario_subj = nazario_raw[['email_id', 'subject']]

enron_subj = pd.read_csv(f'{BASE_PATH}/enron_emails_cleaned_temporal.csv', usecols=['message_id', 'subject'])
enron_subj = enron_subj.rename(columns={'message_id': 'email_id'})

subjects_all = pd.concat([enron_subj, nazario_subj], ignore_index=True).drop_duplicates(subset='email_id')
test_full = test_full.merge(subjects_all, on='email_id', how='left')

test_full['text'] = "[SUBJECT] " + test_full['subject'].fillna('') + " [BODY] " + test_full['body_text'].fillna('')

del nazario_raw, nazario_subj, enron_subj, subjects_all
gc.collect()
print(test_full.shape)

(50646, 38)


In [3]:
@torch.no_grad()
def predict_batch(texts, batch_size=32):
    all_probs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc = tokenizer(batch, truncation=True, max_length=MAX_LENGTH, padding=True, return_tensors='pt').to(device)
        logits = model(**enc).logits
        probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
        all_probs.append(probs)
    return np.concatenate(all_probs)

test_full['phishing_prob'] = predict_batch(test_full['text'].tolist())

THRESHOLD = decision_config['chosen_threshold']
test_full['pred_label'] = (test_full['phishing_prob'] >= THRESHOLD).astype(int)

print(classification_report(test_full['label'], test_full['pred_label'], target_names=['benign','phishing'], digits=4))
print(confusion_matrix(test_full['label'], test_full['pred_label']))

              precision    recall  f1-score   support

      benign     0.9982    0.9995    0.9989     50050
    phishing     0.9513    0.8523    0.8991       596

    accuracy                         0.9977     50646
   macro avg     0.9748    0.9259    0.9490     50646
weighted avg     0.9977    0.9977    0.9977     50646

[[50024    26]
 [   88   508]]


In [4]:
false_negatives = test_full[(test_full['label'] == 1) & (test_full['pred_label'] == 0)]
false_positives = test_full[(test_full['label'] == 0) & (test_full['pred_label'] == 1)]

print(f"False negatives (missed phishing): {len(false_negatives)}")
print(f"False positives (benign flagged as phishing): {len(false_positives)}")

False negatives (missed phishing): 88
False positives (benign flagged as phishing): 26


In [5]:
print("=== False Negatives: model confidence (should be near threshold, not near 0) ===")
print(false_negatives['phishing_prob'].describe())

print("\n=== False Negatives: text length vs. correctly-caught phishing ===")
true_positives = test_full[(test_full['label'] == 1) & (test_full['pred_label'] == 1)]
print("FN body length:", false_negatives['body_text'].str.len().describe()[['mean','50%','max']])
print("TP body length:", true_positives['body_text'].str.len().describe()[['mean','50%','max']])

print("\n=== False Positives: confidence ===")
print(false_positives['phishing_prob'].describe())

=== False Negatives: model confidence (should be near threshold, not near 0) ===
count    88.000000
mean      0.006389
std       0.015711
min       0.000082
25%       0.000125
50%       0.000445
75%       0.004020
max       0.082031
Name: phishing_prob, dtype: float64

=== False Negatives: text length vs. correctly-caught phishing ===
FN body length: mean     1899.579545
50%       994.000000
max     30515.000000
Name: body_text, dtype: float64
TP body length: mean    1.093162e+04
50%     9.245000e+02
max     4.599644e+06
Name: body_text, dtype: float64

=== False Positives: confidence ===
count    26.000000
mean      0.773364
std       0.338004
min       0.102656
25%       0.518391
50%       0.981267
75%       0.999490
max       0.999924
Name: phishing_prob, dtype: float64


In [6]:
error_analysis_cols = ['domain_age_days', 'domain_reputation_score', 'has_dmarc_record',
                        'ip_reputation_score', 'url_count', 'phishing_prob']

print("=== False Negatives (missed phishing) — synthetic feature profile ===")
print(false_negatives[error_analysis_cols].describe())

print("\n=== True Positives (caught phishing) — synthetic feature profile, for comparison ===")
print(true_positives[error_analysis_cols].describe())

=== False Negatives (missed phishing) — synthetic feature profile ===
       domain_age_days  domain_reputation_score  ip_reputation_score  \
count        88.000000                88.000000            88.000000   
mean       1624.070374                51.971396            43.643546   
std        1801.310946                32.145069            25.286171   
min           7.100790                 0.000000             0.000000   
25%         264.619694                16.656244            26.811665   
50%        1529.058287                69.947672            43.677720   
75%        1529.058287                69.947672            65.991131   
max        7622.929769                90.472011            95.386152   

       url_count  phishing_prob  
count  88.000000      88.000000  
mean    0.875000       0.006389  
std     1.760437       0.015711  
min     0.000000       0.000082  
25%     0.000000       0.000125  
50%     0.000000       0.000445  
75%     1.000000       0.004020  
max    11

In [7]:
review_cols = ['email_id', 'subject', 'body_text', 'label', 'pred_label', 'phishing_prob',
               'domain_age_days', 'domain_reputation_score', 'has_dmarc_record']

false_negatives[review_cols].to_csv(f'{BASE_PATH}/false_negatives_review.csv', index=False)
false_positives[review_cols].to_csv(f'{BASE_PATH}/false_positives_review.csv', index=False)

print("Saved false_negatives_review.csv and false_positives_review.csv for manual read-through.")

Saved false_negatives_review.csv and false_positives_review.csv for manual read-through.
